# Incident Response Runbook: Collection - Screen Capture

**Tactic:** Collection
**Technique:** Screen Capture (T1113)
**Severity:** HIGH

## Overview

This runbook provides step-by-step incident response procedures for Screen Capture activities.

## Incident Response Phases

1. **Detection & Analysis**
2. **Containment**
3. **Eradication**
4. **Recovery**
5. **Post-Incident Activities**


## Phase 1: Detection & Analysis

### Objectives
- Confirm the incident
- Identify the scope
- Assess impact


In [ ]:
import json
import re
from datetime import datetime
import sys
import os

# Add the project root to the Python path
sys.path.append(os.path.abspath(os.path.join(os.path.dirname(__file__), '..', '..')))

from splunk.splunk_data_collector import SplunkDataCollector
from crowdstrike.crowdstrike_response import CrowdStrikeResponse
from iris.iris_integration import IRISIntegration
from misp.misp_integration import MISPIntegration
from shuffle.shuffle_integration import ShuffleIntegration

# Initialize integrations
splunk = SplunkDataCollector()
crowdstrike = CrowdStrikeResponse()
iris = IRISIntegration()
misp = MISPIntegration()
shuffle = ShuffleIntegration()

# Step 1: Detection & Analysis
print("=" * 60)
print("STEP 1: Detection & Analysis")
print("=" * 60)

detection_time = datetime.now().isoformat()

# Query Splunk for screen capture indicators
print(f"\n[QUERY] Searching Splunk for screen capture indicators...")
splunk_query = '''
index=* (sourcetype=WinEventLog:Security EventCode=4688)
OR (sourcetype=linux_secure "scrot" OR "import" OR "xwd" OR "screencapture")
OR (sourcetype=process_monitor "screencapture" OR "screenshot" OR "screengrab")
| eval command=coalesce(CommandLine, cmd, command)
| where match(command, "screencapture|scrot|import|xwd|screenshot|screengrab|Get-Screenshot|PrintWindow")
| stats count by host, user, command, _time
| where count > 2
'''

try:
    splunk_results = splunk.search_events(splunk_query, timeframe="-24h")
    print(f"   Found {len(splunk_results)} potential screen capture events")
except Exception as e:
    print(f"   Splunk query failed: {e}")
    splunk_results = []

# Extract affected systems and screen capture activities
affected_systems = []
screen_capture_indicators = []
unique_users = set()

for event in splunk_results:
    system_info = {
        'hostname': event.get('host', 'unknown'),
        'user': event.get('user', 'unknown'),
        'command': event.get('command', ''),
        'event_count': event.get('count', 0),
        'last_seen': event.get('_time', detection_time)
    }
    affected_systems.append(system_info)
    unique_users.add(event.get('user', 'unknown'))

    # Extract indicators
    if event.get('command'):
        screen_capture_indicators.append({
            'type': 'screen_capture_command',
            'value': event.get('command'),
            'context': f"Executed by {event.get('user', 'unknown')} on {event.get('host', 'unknown')}"
        })

# Query CrowdStrike for endpoint detection
print(f"\n[QUERY] Checking CrowdStrike for screen capture detections...")
try:
    cs_detections = crowdstrike.get_detections(
        filter="technique:'Screen Capture'",
        start_time="-24h"
    )
    cs_affected_hosts = []
    for detection in cs_detections:
        host_info = {
            'device_id': detection.get('device_id', ''),
            'hostname': detection.get('hostname', ''),
            'detection_id': detection.get('detection_id', ''),
            'technique': detection.get('technique', ''),
            'severity': detection.get('severity', 0)
        }
        cs_affected_hosts.append(host_info)

        # Merge with Splunk data
        existing_system = next((s for s in affected_systems if s['hostname'] == host_info['hostname']), None)
        if existing_system:
            existing_system.update(host_info)
        else:
            affected_systems.append(host_info)

    print(f"   Found {len(cs_affected_hosts)} CrowdStrike detections")
except Exception as e:
    print(f"   CrowdStrike query failed: {e}")
    cs_affected_hosts = []

# Enrich with threat intelligence from MISP
print(f"\n[ENRICHMENT] Checking MISP for related threat intelligence...")
misp_results = []
try:
    for indicator in screen_capture_indicators:
        misp_search = misp.search_iocs(indicator['value'])
        if misp_search:
            misp_results.extend(misp_search)
            indicator['threat_intel'] = misp_search[0] if misp_search else None
            print(f"   Found threat intelligence for {indicator['value']}")
except Exception as e:
    print(f"   MISP enrichment failed: {e}")

# Create IRIS case
print(f"\n[CASE] Creating IRIS incident case...")
try:
    incident_data = {
        'title': f'Screen Capture Collection Incident - {len(affected_systems)} systems',
        'description': f'Detected screen capture collection activities on {len(affected_systems)} systems',
        'severity': 'HIGH',
        'tactic': 'Collection',
        'technique': 'Screen Capture (T1113)',
        'indicators': screen_capture_indicators,
        'affected_systems': affected_systems,
        'threat_intelligence': misp_results
    }

    incident_id = iris.create_case(incident_data)
    if incident_id:
        print(f"   Created IRIS case: {incident_id}")
    else:
        print(f"   Failed to create IRIS case")
        incident_id = f"LOCAL-{datetime.now().strftime('%Y%m%d-%H%M%S')}"

except Exception as e:
    print(f"   IRIS case creation failed: {e}")
    incident_id = f"LOCAL-{datetime.now().strftime('%Y%m%d-%H%M%S')}"

print(f"\n Detection complete:")
print(f"  - Affected systems: {len(affected_systems)}")
print(f"  - Unique users: {len(unique_users)}")
print(f"  - Screen capture indicators: {len(screen_capture_indicators)}")
print(f"  - Threat intelligence hits: {len(misp_results)}")
print(f"  - Incident ID: {incident_id}")


## Phase 2: Containment

### Objectives
- Stop the attack
- Isolate affected systems
- Prevent spread


In [ ]:
# Step 2: Containment
print("\n" + "=" * 60)
print("STEP 2: Containment")
print("=" * 60)

containment_time = datetime.now().isoformat()
containment_actions = []
isolated_hosts = []
disabled_accounts = []
blocked_ips = []

# 1. Isolate affected systems
print(f"\n[CONTAINMENT] Isolating affected systems...")
try:
    for system in affected_systems:
        if system.get('device_id'):
            # Use CrowdStrike to isolate host
            isolation_result = crowdstrike.isolate_host(system['device_id'])
            if isolation_result:
                isolated_hosts.append(system['hostname'])
                containment_actions.append({
                    'action': 'host_isolation',
                    'target': system['hostname'],
                    'method': 'CrowdStrike',
                    'status': 'success',
                    'timestamp': containment_time
                })
                print(f"   Isolated host: {system['hostname']}")
            else:
                print(f"   Failed to isolate host: {system['hostname']}")
        else:
            # Use Shuffle for network isolation
            network_isolation = shuffle.isolate_system(system['hostname'])
            if network_isolation:
                isolated_hosts.append(system['hostname'])
                containment_actions.append({
                    'action': 'network_isolation',
                    'target': system['hostname'],
                    'method': 'Shuffle',
                    'status': 'success',
                    'timestamp': containment_time
                })
                print(f"   Network isolated: {system['hostname']}")
except Exception as e:
    print(f"   Error during host isolation: {e}")

# 2. Disable suspicious accounts
print(f"\n[CONTAINMENT] Disabling suspicious accounts...")
try:
    for user in unique_users:
        if user and user != 'unknown':
            # Disable account via Shuffle
            disable_result = shuffle.disable_account(user)
            if disable_result:
                disabled_accounts.append(user)
                containment_actions.append({
                    'action': 'account_disable',
                    'target': user,
                    'method': 'Shuffle',
                    'status': 'success',
                    'timestamp': containment_time
                })
                print(f"   Disabled account: {user}")
except Exception as e:
    print(f"   Error disabling accounts: {e}")

# 3. Block suspicious IPs/domains
print(f"\n[CONTAINMENT] Blocking suspicious IPs and domains...")
try:
    suspicious_ips = []
    for indicator in screen_capture_indicators:
        # Extract potential IPs from commands or context
        ip_pattern = r'\b\d{1,3}\.\d{1,3}\.\d{1,3}\.\d{1,3}\b'
        ips = re.findall(ip_pattern, indicator.get('context', ''))
        suspicious_ips.extend(ips)

    for ip in set(suspicious_ips):
        block_result = shuffle.block_ip(ip)
        if block_result:
            blocked_ips.append(ip)
            containment_actions.append({
                'action': 'ip_block',
                'target': ip,
                'method': 'Shuffle',
                'status': 'success',
                'timestamp': containment_time
            })
            print(f"   Blocked IP: {ip}")
except Exception as e:
    print(f"   Error blocking IPs: {e}")

# 4. Enable enhanced monitoring
print(f"\n[CONTAINMENT] Enabling enhanced monitoring...")
try:
    monitoring_rules = [
        {
            'name': 'Enhanced Screen Capture Monitoring',
            'query': 'index=* EventCode=4688 | eval risk_score = case(match(CommandLine, "screencapture|scrot|import|xwd"), 9, match(CommandLine, "screenshot|screengrab"), 8, 1==1, 4) | where risk_score >= 7',
            'alert_threshold': 1,
            'time_window': '5m'
        }
    ]

    enhanced_monitoring = splunk.enable_enhanced_monitoring(monitoring_rules)
    if enhanced_monitoring:
        containment_actions.append({
            'action': 'enhanced_monitoring',
            'target': 'Splunk',
            'method': 'Splunk',
            'status': 'success',
            'timestamp': containment_time
        })
        print(f"   Enabled enhanced monitoring in Splunk")
except Exception as e:
    print(f"   Error enabling enhanced monitoring: {e}")

print(f"\n Containment complete:")
print(f"  - Hosts isolated: {len(isolated_hosts)}")
print(f"  - Accounts disabled: {len(disabled_accounts)}")
print(f"  - IPs blocked: {len(blocked_ips)}")
print(f"  - Enhanced monitoring: enabled")


## Phase 3: Eradication

### Objectives
- Remove malicious artifacts
- Close vulnerabilities
- Verify threat removal


In [ ]:
# Step 3: Eradication
print("\n" + "=" * 60)
print("STEP 3: Eradication")
print("=" * 60)

eradication_time = datetime.now().isoformat()
eradication_actions = []
terminated_processes = []
deleted_tools = []
reset_credentials = []

# 1. Terminate screen capture processes
print(f"\n[ERADICATION] Terminating screen capture processes...")
try:
    for system in affected_systems:
        if system.get('device_id'):
            # Get running processes from CrowdStrike
            processes = crowdstrike.get_processes(system['device_id'])
            capture_procs = []

            for proc in processes:
                cmd_line = proc.get('command_line', '').lower()
                if any(capture_cmd in cmd_line for capture_cmd in ['screencapture', 'scrot', 'import', 'xwd', 'screenshot', 'screengrab']):
                    capture_procs.append(proc)

            for proc in capture_procs:
                kill_result = crowdstrike.kill_process(system['device_id'], proc['process_id'])
                if kill_result:
                    terminated_processes.append(f"{system['hostname']}:{proc['process_name']}")
                    eradication_actions.append({
                        'action': 'process_termination',
                        'target': f"{system['hostname']}:{proc['process_name']}",
                        'method': 'CrowdStrike',
                        'status': 'success',
                        'timestamp': eradication_time
                    })
                    print(f"   Terminated screen capture process: {proc['process_name']} on {system['hostname']}")
except Exception as e:
    print(f"   Error terminating processes: {e}")

# 2. Remove screen capture tools
print(f"\n[ERADICATION] Removing screen capture tools...")
try:
    for system in affected_systems:
        if system.get('device_id'):
            # Scan for and remove common screen capture tools
            tools_to_remove = [
                'screencapture.exe', 'scrot.exe', 'import.exe', 'xwd.exe',
                'screenshot.ps1', 'screengrab.bat', 'capture.vbs'
            ]

            for tool in tools_to_remove:
                removal_result = crowdstrike.remove_file(system['device_id'], tool)
                if removal_result:
                    deleted_tools.append(f"{system['hostname']}:{tool}")
                    eradication_actions.append({
                        'action': 'tool_removal',
                        'target': f"{system['hostname']}:{tool}",
                        'method': 'CrowdStrike',
                        'status': 'success',
                        'timestamp': eradication_time
                    })
                    print(f"   Removed screen capture tool: {tool} from {system['hostname']}")
except Exception as e:
    print(f"   Error removing tools: {e}")

# 3. Reset compromised credentials
print(f"\n[ERADICATION] Resetting compromised credentials...")
try:
    for user in unique_users:
        if user and user != 'unknown':
            # Reset password via Shuffle
            reset_result = shuffle.reset_password(user)
            if reset_result:
                reset_credentials.append(user)
                eradication_actions.append({
                    'action': 'credential_reset',
                    'target': user,
                    'method': 'Shuffle',
                    'status': 'success',
                    'timestamp': eradication_time
                })
                print(f"   Reset credentials for: {user}")
except Exception as e:
    print(f"   Error resetting credentials: {e}")

# 4. Clean up captured files
print(f"\n[ERADICATION] Cleaning up captured screenshot files...")
try:
    for system in affected_systems:
        if system.get('device_id'):
            # Remove recently created image files that might be screenshots
            cleanup_result = crowdstrike.cleanup_screenshots(system['device_id'])
            if cleanup_result:
                eradication_actions.append({
                    'action': 'screenshot_cleanup',
                    'target': system['hostname'],
                    'method': 'CrowdStrike',
                    'status': 'success',
                    'timestamp': eradication_time
                })
                print(f"   Cleaned up screenshot files on: {system['hostname']}")
except Exception as e:
    print(f"   Error cleaning up screenshots: {e}")

# 5. Verify eradication
print(f"\n[ERADICATION] Verifying eradication...")
try:
    verification_results = []
    for system in affected_systems:
        if system.get('device_id'):
            # Run verification scan for screen capture indicators
            scan_result = crowdstrike.scan_system(system['device_id'], 'screen_capture_indicators')
            verification_results.append({
                'system': system['hostname'],
                'scan_result': scan_result,
                'clean': scan_result.get('screen_capture_tools_found', 0) == 0
            })

    clean_systems = [r for r in verification_results if r['clean']]
    eradication_actions.append({
        'action': 'eradication_verification',
        'target': f"{len(clean_systems)}/{len(verification_results)} systems clean",
        'method': 'CrowdStrike',
        'status': 'success' if len(clean_systems) == len(verification_results) else 'partial',
        'timestamp': eradication_time
    })

    print(f"   Verification complete: {len(clean_systems)}/{len(verification_results)} systems clean")

except Exception as e:
    print(f"   Error during verification: {e}")

print(f"\n Eradication complete:")
print(f"  - Processes terminated: {len(terminated_processes)}")
print(f"  - Tools deleted: {len(deleted_tools)}")
print(f"  - Credentials reset: {len(reset_credentials)}")
print(f"  - Screenshot files cleaned: ")
print(f"  - Systems verified clean: {len([r for r in verification_results if r.get('clean', False)])}")


## Phase 4: Recovery

### Objectives
- Restore systems
- Validate functionality
- Monitor for issues


In [ ]:
# Step 4: Recovery
print("\n" + "=" * 60)
print("STEP 4: Recovery")
print("=" * 60)

recovery_time = datetime.now().isoformat()
recovery_actions = []
reenabled_hosts = []
restored_accounts = []

# 1. Re-enable isolated systems
print(f"\n[RECOVERY] Re-enabling isolated systems...")
try:
    for host in isolated_hosts:
        system = next((s for s in affected_systems if s['hostname'] == host), None)
        if system and system.get('device_id'):
            # Use CrowdStrike to re-enable host
            reenable_result = crowdstrike.reenable_host(system['device_id'])
            if reenable_result:
                reenabled_hosts.append(host)
                recovery_actions.append({
                    'action': 'host_reenable',
                    'target': host,
                    'method': 'CrowdStrike',
                    'status': 'success',
                    'timestamp': recovery_time
                })
                print(f"   Re-enabled host: {host}")
            else:
                print(f"   Failed to re-enable host: {host}")
        else:
            # Use Shuffle for network re-enablement
            network_restore = shuffle.restore_system(host)
            if network_restore:
                reenabled_hosts.append(host)
                recovery_actions.append({
                    'action': 'network_restore',
                    'target': host,
                    'method': 'Shuffle',
                    'status': 'success',
                    'timestamp': recovery_time
                })
                print(f"   Restored network access: {host}")
except Exception as e:
    print(f"   Error re-enabling systems: {e}")

# 2. Restore disabled accounts
print(f"\n[RECOVERY] Restoring disabled accounts...")
try:
    for user in disabled_accounts:
        # Restore account access via Shuffle
        restore_result = shuffle.restore_account(user)
        if restore_result:
            restored_accounts.append(user)
            recovery_actions.append({
                'action': 'account_restore',
                'target': user,
                'method': 'Shuffle',
                'status': 'success',
                'timestamp': recovery_time
            })
            print(f"   Restored account: {user}")
except Exception as e:
    print(f"   Error restoring accounts: {e}")

# 3. Validate system functionality
print(f"\n[RECOVERY] Validating system functionality...")
try:
    for system in affected_systems:
        if system.get('device_id'):
            # Validate system health after recovery
            health_check = crowdstrike.validate_system_health(system['device_id'])
            if health_check and health_check.get('status') == 'healthy':
                recovery_actions.append({
                    'action': 'system_validation',
                    'target': system['hostname'],
                    'method': 'CrowdStrike',
                    'status': 'success',
                    'timestamp': recovery_time
                })
                print(f"   System health validated: {system['hostname']}")
            else:
                print(f"   System health issues detected: {system['hostname']}")
except Exception as e:
    print(f"   Error validating system health: {e}")

# 4. Restore monitoring and alerting
print(f"\n[RECOVERY] Restoring monitoring and alerting...")
try:
    # Restore normal Splunk monitoring rules
    normal_monitoring = splunk.restore_normal_monitoring()
    if normal_monitoring:
        recovery_actions.append({
            'action': 'monitoring_restore',
            'target': 'Splunk',
            'method': 'Splunk',
            'status': 'success',
            'timestamp': recovery_time
        })
        print(f"   Restored normal monitoring in Splunk")

    # Restore CrowdStrike normal operations
    cs_normal_ops = crowdstrike.restore_normal_operations()
    if cs_normal_ops:
        recovery_actions.append({
            'action': 'cs_normal_ops',
            'target': 'CrowdStrike',
            'method': 'CrowdStrike',
            'status': 'success',
            'timestamp': recovery_time
        })
        print(f"   Restored normal CrowdStrike operations")

except Exception as e:
    print(f"   Error restoring monitoring: {e}")

# 5. Verify data integrity
print(f"\n[RECOVERY] Verifying data integrity...")
try:
    integrity_checks = []
    for system in affected_systems:
        # Check that no unauthorized screen captures occurred during incident
        integrity_result = shuffle.verify_no_unauthorized_captures(system['hostname'])
        integrity_checks.append({
            'system': system['hostname'],
            'integrity_ok': integrity_result.get('no_unauthorized_captures', True),
            'captures_detected': integrity_result.get('recent_captures', 0)
        })

    systems_with_integrity = [c for c in integrity_checks if c['integrity_ok']]
    recovery_actions.append({
        'action': 'data_integrity_check',
        'target': f"{len(systems_with_integrity)}/{len(integrity_checks)} systems with verified integrity",
        'method': 'Shuffle',
        'status': 'success' if len(systems_with_integrity) == len(integrity_checks) else 'partial',
        'timestamp': recovery_time
    })

    print(f"   Data integrity check complete: {len(systems_with_integrity)}/{len(integrity_checks)} systems OK")

except Exception as e:
    print(f"   Error verifying data integrity: {e}")

print(f"\n Recovery complete:")
print(f"  - Hosts re-enabled: {len(reenabled_hosts)}")
print(f"  - Accounts restored: {len(restored_accounts)}")
print(f"  - Systems validated: ")
print(f"  - Data integrity verified: {len([c for c in integrity_checks if c.get('integrity_ok', False)])} systems")


## Phase 5: Post-Incident Activities

### Objectives
- Document findings
- Implement improvements
- Share lessons learned


In [ ]:
# Step 5: Post-Incident Actions
print("\n" + "=" * 60)
print("STEP 5: Post-Incident Actions")
print("=" * 60)

post_incident_time = datetime.now().isoformat()
post_incident_actions = []

# 1. Generate comprehensive incident report
print(f"\n[POST-INCIDENT] Generating incident report...")
try:
    incident_report = {
        'incident_id': incident_id,
        'technique': 'T1113 - Screen Capture',
        'detection_time': detection_time,
        'containment_time': containment_time,
        'eradication_time': eradication_time,
        'recovery_time': recovery_time,
        'post_incident_time': post_incident_time,
        'affected_systems': len(affected_systems),
        'isolated_hosts': len(isolated_hosts),
        'disabled_accounts': len(disabled_accounts),
        'terminated_processes': len(terminated_processes),
        'deleted_tools': len(deleted_tools),
        'reenabled_hosts': len(reenabled_hosts),
        'restored_accounts': len(restored_accounts),
        'threat_indicators': len(threat_indicators),
        'timeline': {
            'detection': detection_time,
            'containment': containment_time,
            'eradication': eradication_time,
            'recovery': recovery_time,
            'closure': post_incident_time
        },
        'actions_taken': {
            'detection': detection_actions,
            'containment': containment_actions,
            'eradication': eradication_actions,
            'recovery': recovery_actions
        },
        'recommendations': [
            'Implement stricter screen capture monitoring',
            'Review and update endpoint protection policies',
            'Conduct user awareness training on screen capture risks',
            'Enhance network segmentation for sensitive systems',
            'Regular security assessments for screen capture tools'
        ]
    }

    # Save report to IRIS
    iris_report = iris.create_incident_report(incident_report)
    if iris_report:
        post_incident_actions.append({
            'action': 'incident_report',
            'target': 'IRIS',
            'method': 'IRIS',
            'status': 'success',
            'timestamp': post_incident_time
        })
        print(f"   Incident report saved to IRIS: {iris_report.get('report_id', 'N/A')}")

    # Share threat intelligence with MISP
    misp_sharing = misp.share_threat_intelligence(threat_indicators, incident_report)
    if misp_sharing:
        post_incident_actions.append({
            'action': 'threat_intel_sharing',
            'target': 'MISP',
            'method': 'MISP',
            'status': 'success',
            'timestamp': post_incident_time
        })
        print(f"   Threat intelligence shared with MISP community")

except Exception as e:
    print(f"   Error generating incident report: {e}")

# 2. Conduct lessons learned analysis
print(f"\n[POST-INCIDENT] Conducting lessons learned analysis...")
try:
    lessons_learned = {
        'incident_type': 'Screen Capture Collection (T1113)',
        'root_cause': 'Unauthorized screen capture tools deployed on endpoints',
        'detection_effectiveness': 'High - Automated detection via Splunk and CrowdStrike',
        'response_effectiveness': 'High - Automated containment and eradication successful',
        'recovery_effectiveness': 'High - Systems restored with integrity verification',
        'gaps_identified': [
            'Need enhanced monitoring for screen capture applications',
            'Consider implementing application whitelisting',
            'Improve user training on screen capture risks'
        ],
        'improvements_needed': [
            'Update detection rules for new screen capture techniques',
            'Implement automated response for similar threats',
            'Enhance endpoint visibility and control'
        ],
        'preventive_measures': [
            'Deploy application control policies',
            'Implement enhanced monitoring for screen capture activities',
            'Regular security awareness training',
            'Update incident response procedures'
        ]
    }

    # Document lessons learned in IRIS
    lessons_doc = iris.document_lessons_learned(lessons_learned)
    if lessons_doc:
        post_incident_actions.append({
            'action': 'lessons_learned',
            'target': 'IRIS',
            'method': 'IRIS',
            'status': 'success',
            'timestamp': post_incident_time
        })
        print(f"   Lessons learned documented in IRIS")

except Exception as e:
    print(f"   Error conducting lessons learned: {e}")

# 3. Implement preventive measures
print(f"\n[POST-INCIDENT] Implementing preventive measures...")
try:
    preventive_actions = []

    # Update Splunk monitoring rules
    enhanced_monitoring = splunk.implement_enhanced_monitoring('screen_capture')
    if enhanced_monitoring:
        preventive_actions.append('Enhanced Splunk monitoring for screen capture')
        print(f"   Enhanced Splunk monitoring rules implemented")

    # Update CrowdStrike policies
    policy_updates = crowdstrike.update_policies('screen_capture_prevention')
    if policy_updates:
        preventive_actions.append('Updated CrowdStrike endpoint policies')
        print(f"   CrowdStrike policies updated for screen capture prevention")

    # Implement Shuffle automated responses
    auto_response = shuffle.implement_automated_response('screen_capture')
    if auto_response:
        preventive_actions.append('Automated response workflows implemented')
        print(f"   Automated response workflows implemented in Shuffle")

    post_incident_actions.append({
        'action': 'preventive_measures',
        'target': preventive_actions,
        'method': 'Multiple',
        'status': 'success',
        'timestamp': post_incident_time
    })

except Exception as e:
    print(f"   Error implementing preventive measures: {e}")

# 4. Update security policies and procedures
print(f"\n[POST-INCIDENT] Updating security policies...")
try:
    policy_updates = {
        'screen_capture_policy': {
            'description': 'Enhanced screen capture monitoring and control',
            'changes': [
                'Added monitoring for screen capture applications',
                'Implemented automated response for unauthorized captures',
                'Enhanced endpoint protection policies'
            ]
        },
        'incident_response_procedure': {
            'description': 'Updated IR procedures for screen capture incidents',
            'changes': [
                'Added automated containment steps',
                'Enhanced eradication procedures',
                'Improved recovery validation'
            ]
        }
    }

    # Update policies in Shuffle (workflow management)
    policy_update_result = shuffle.update_security_policies(policy_updates)
    if policy_update_result:
        post_incident_actions.append({
            'action': 'policy_updates',
            'target': 'Shuffle',
            'method': 'Shuffle',
            'status': 'success',
            'timestamp': post_incident_time
        })
        print(f"   Security policies updated in Shuffle")

except Exception as e:
    print(f"   Error updating security policies: {e}")

# 5. Close incident case
print(f"\n[POST-INCIDENT] Closing incident case...")
try:
    closure_summary = {
        'incident_id': incident_id,
        'closure_time': post_incident_time,
        'resolution_status': 'Resolved - Threat contained and eradicated',
        'final_status': 'Closed',
        'total_actions': len(detection_actions) + len(containment_actions) + len(eradication_actions) + len(recovery_actions) + len(post_incident_actions),
        'affected_assets': len(affected_systems),
        'threat_actor': 'Unknown - Automated response prevented further activity',
        'data_compromised': 'Screen capture data - contained before exfiltration',
        'business_impact': 'Minimal - Automated response prevented data loss'
    }

    # Close case in IRIS
    case_closure = iris.close_incident_case(incident_id, closure_summary)
    if case_closure:
        post_incident_actions.append({
            'action': 'case_closure',
            'target': f"IRIS Case {incident_id}",
            'method': 'IRIS',
            'status': 'success',
            'timestamp': post_incident_time
        })
        print(f"   Incident case closed in IRIS: {incident_id}")

    # Archive incident data
    archive_result = shuffle.archive_incident_data(incident_id)
    if archive_result:
        print(f"   Incident data archived for compliance and analysis")

except Exception as e:
    print(f"   Error closing incident case: {e}")

print(f"\n Post-incident actions complete:")
print(f"  - Incident report generated: ")
print(f"  - Lessons learned documented: ")
print(f"  - Preventive measures implemented: ")
print(f"  - Security policies updated: ")
print(f"  - Incident case closed: ")
print(f"\n Screen Capture Collection Incident Response Complete")
print(f"   Incident ID: {incident_id}")
print(f"   Total Response Time: Automated (minutes)")
print(f"   Systems Protected: {len(affected_systems)}")
print(f"   Threat Contained: ")


## Summary

This incident response runbook has guided you through the complete lifecycle of responding to this incident.

### Key Takeaways
- Rapid detection and response are critical
- Containment prevents further damage
- Thorough eradication prevents reinfection
- Continuous monitoring ensures recovery

### Next Steps
- Review and approve the incident report
- Implement recommended preventive measures
- Schedule follow-up review
- Share lessons learned with the team
